# Natural Gas Futures Weekly Data: Frequency Component Analysis
This notebook analyzes the frequency components of natural gas futures weekly prices using the Fast Fourier Transform (FFT). We extract and visualize low, medium, and high frequency components.

In [ ]:
import csv
from numpy.fft import fft, ifft, fftfreq
from numpy import real, imag
import matplotlib.pyplot as plt

# Read the data
weekly_prices = []
dates = []
with open('natural_gas_futures_weekly_all.csv','r') as file:
    csv_handle = csv.DictReader(file)
    for rows in csv_handle:
        dates.append(rows['Date'])
        weekly_prices.append(0.5 * (float(rows['High']) + float(rows['Low'])))

In [ ]:
# Plot the original weekly prices
plt.figure()
plt.plot(range(len(weekly_prices)), weekly_prices, '-b')
plt.xlabel('Week #')
plt.ylabel('Crude Oil Future Price')
plt.title('Weekly Natural Gas Futures Prices')
plt.show()

In [ ]:
# Compute FFT and frequencies
fft_data = fft(weekly_prices)
N = len(fft_data)
fft_frequencies = fftfreq(N, d=1)  # d=1 because data is weekly

In [ ]:
# Function to select all items in a frequency range
def select_all_items_in_freq_range(lo, hi):
    new_fft_data = []
    for (fft_val, fft_freq) in zip(fft_data, fft_frequencies):
        if lo <= abs(fft_freq) < hi:
            new_fft_data.append(fft_val)
        else:
            new_fft_data.append(0.0)
    filtered_data = ifft(new_fft_data)
    assert all(abs(imag(x)) <= 1E-10 for x in filtered_data)
    return [real(x) for x in filtered_data]

In [ ]:
# Extract the required frequency bands
upto_1_year = select_all_items_in_freq_range(0, 1/52)
one_year_to_1_quarter = select_all_items_in_freq_range(1/52, 1/13)
less_than_1_quarter = select_all_items_in_freq_range(1/13, 0.5)

In [ ]:
# Plot frequency components < once/year
plt.figure()
plt.plot(upto_1_year, '-b', lw=2, label='Low frequency (<1/year)')
plt.plot(weekly_prices, '--r', lw=0.2, label='Original')
plt.xlabel('Week #')
plt.ylabel('Price')
plt.title('Frequency components < once/year')
plt.legend()
plt.show()

In [ ]:
# Plot frequency components between once/year and once/quarter
plt.figure()
plt.plot(one_year_to_1_quarter, '-b', lw=2, label='Medium frequency (1/year to 1/quarter)')
plt.plot(weekly_prices, '--r', lw=0.2, label='Original')
plt.title('Frequency components between once/year and once/quarter')
plt.xlabel('Week #')
plt.ylabel('Price')
plt.legend()
plt.show()

In [ ]:
# Plot frequency components > once/quarter
plt.figure()
plt.plot(less_than_1_quarter, '-b', lw=2, label='High frequency (>1/quarter)')
plt.plot(weekly_prices, '--r', lw=0.2, label='Original')
plt.title('Frequency components > once/quarter')
plt.xlabel('Week #')
plt.ylabel('Price')
plt.legend()
plt.show()

In [ ]:
# Plot sum of all components vs original
plt.figure()
reconstructed = [v1 + v2 + v3 for (v1, v2, v3) in zip(upto_1_year, one_year_to_1_quarter, less_than_1_quarter)]
plt.plot(reconstructed, '-b', lw=2, label='Sum of components')
plt.plot(weekly_prices, '--r', lw=0.2, label='Original')
plt.title('Sum of all the components')
plt.xlabel('Week #')
plt.ylabel('Prices')
plt.legend()
plt.show()

In [ ]:
# Sanity checks
N = len(weekly_prices)
assert(len(fft_frequencies) == len(weekly_prices))
assert(fft_frequencies[0] == 0.0)
assert(abs(fft_frequencies[N//2] - 0.5 ) <= 0.05), f'fft frequncies incorrect: {fft_frequencies[N//2]} does not equal 0.5'
assert(abs(fft_frequencies[N//4] - 0.25 ) <= 0.05), f'fft frequncies incorrect:  {fft_frequencies[N//4]} does not equal 0.25'
assert(abs(fft_frequencies[3*N//4] + 0.25 ) <= 0.05), f'fft frequncies incorrect:  {fft_frequencies[3*N//4]} does not equal -0.25'
assert(abs(fft_frequencies[1] - 1/N ) <= 0.05), f'fft frequncies incorrect:  {fft_frequencies[1]} does not equal {1/N}'
assert(abs(fft_frequencies[N-1] + 1/N ) <= 0.05), f'fft frequncies incorrect:  {fft_frequencies[N-1]} does not equal {-1/N}'

for (v1, v2, v3, v4) in zip(weekly_prices, upto_1_year, one_year_to_1_quarter, less_than_1_quarter ):
    assert ( abs(v1 - (v2 + v3 + v4)) <= 0.01), 'The components are not adding up -- there is a mistake in the way you split your original signal into various components'
print('All tests OK -- 10 points!!')